## 1. Introduction
Optimization algorithms aim to find the minimum (or maximum) of an objective function. Among these methods, Newton's method and its variants are particularly powerful for twice-differentiable functions. This document introduces Newton's method and then presents a systematic derivation of the BFGS method (named after Broyden, Fletcher, Goldfarb, and Shanno), which is a popular quasi-Newton optimization algorithm.

## 2. Newton's Method

### 2.1 Principle
Newton's method directly uses the second derivatives (Hessian) of the objective function to find its minimum. It's based on a quadratic approximation of the function around the current point.

### 2.2 Taylor Series Approximation
For a twice-differentiable function $f(x)$, we can approximate it around a point $x_k$ using Taylor series:

$f(x_k + p) \approx f(x_k) + \nabla f(x_k)^T p + \frac{1}{2}p^T \nabla^2 f(x_k) p$

Where:
- $f(x)$ is the objective function we want to minimize
- $x_k$ is the current point (iterate) at iteration $k$
- $p$ is a step direction vector from $x_k$
- $\nabla f(x_k)$ is the gradient of $f$ at $x_k$
- $\nabla^2 f(x_k)$ is the Hessian matrix (matrix of second derivatives) at $x_k$

### 2.3 Finding the Minimum
To find the minimum of this quadratic approximation, we set its gradient to zero:

$\nabla f(x_k) + \nabla^2 f(x_k) p = 0$

Solving for $p$:

$p = -[\nabla^2 f(x_k)]^{-1} \nabla f(x_k)$

This gives us the Newton step.

### 2.4 Iteration Process
The Newton method iterates:

$x_{k+1} = x_k - [\nabla^2 f(x_k)]^{-1} \nabla f(x_k)$

Where:
- $x_{k+1}$ is the next iterate
- $[\nabla^2 f(x_k)]^{-1}$ is the inverse of the Hessian matrix at $x_k$

### 2.5 Newton's Method Algorithm: Step-by-Step

**Initialization:**
- Start with initial point $x_0$
- Set tolerance $\epsilon > 0$
- Set iteration counter $k = 0$

**Main Loop:**
1. **Compute gradient**: $\nabla f_k = \nabla f(x_k)$
2. **Compute Hessian**: $H_k = \nabla^2 f(x_k)$
3. **Check if Hessian is positive definite**:
   - If not, modify $H_k$ to make it positive definite (e.g., add $\delta I$ where $\delta > 0$)
4. **Compute Newton direction**: $p_k = -H_k^{-1} \nabla f_k$
5. **Determine step length**: 
   - Pure Newton: $\alpha_k = 1$
   - Damped Newton: Find $\alpha_k$ by line search
6. **Update position**: $x_{k+1} = x_k + \alpha_k p_k$
7. **Check convergence**: If $\|\nabla f_k\| < \epsilon$ or $\|x_{k+1} - x_k\| < \epsilon$, stop
8. **Increment counter**: $k = k + 1$
9. **Return to step 1**

### 2.6 Practical Implementation Details

1. **Computing the Hessian**:
   - For small problems, analytical formulas can be used
   - For larger problems, finite differences or automatic differentiation is often employed
   - The Hessian computation is often the most expensive step

2. **Solving the Newton system**:
   - Direct methods (e.g., Cholesky decomposition) for smaller problems
   - Iterative methods (e.g., conjugate gradient) for larger problems

3. **Ensuring positive definiteness**:
   - Modified Newton methods add a positive multiple of the identity matrix to the Hessian if needed
   - This ensures the search direction is a descent direction

4. **Line search**:
   - Pure Newton uses $\alpha_k = 1$, which works well near the solution
   - Damped Newton uses line search to ensure descent, especially far from the solution

### 2.7 Limitations of Newton's Method
Newton's method has several drawbacks:
1. Computing the Hessian and its inverse can be computationally expensive
2. The Hessian may not be positive definite, leading to steps that don't decrease the function
3. Storing the full Hessian requires $O(n^2)$ memory for $n$ variables

These limitations motivate the development of quasi-Newton methods like BFGS, which approximate the Hessian or its inverse.

## 3. BFGS Method: Quadratic Model via Taylor Series Expansion

### 3.1 Taylor Series Approximation
We begin by approximating the objective function $f(x)$ near the current iterate $x_k$ using a second-order Taylor series expansion:

$$f(x_k + p) \approx f(x_k) + \nabla f(x_k)^T p + \frac{1}{2}p^T \nabla^2 f(x_k) p$$

Where:
- $f(x)$ is the objective function we want to minimize
- $x_k$ is the current point (iterate) at iteration $k$
- $p$ is a step direction vector from $x_k$
- $\nabla f(x_k)$ is the gradient of $f$ at $x_k$
- $\nabla^2 f(x_k)$ is the Hessian matrix (matrix of second derivatives) at $x_k$

In quasi-Newton methods, we don't compute the Hessian $\nabla^2 f(x_k)$ directly. Instead, we use an approximation $B_k$, giving us the quadratic model:

$$m_k(p) = f_k + \nabla f_k^T p + \frac{1}{2}p^T B_k p \quad (6.1)$$

Where:
- $m_k(p)$ is our quadratic model of the objective function at iteration $k$
- $f_k = f(x_k)$ is the function value at the current iterate
- $\nabla f_k = \nabla f(x_k)$ is the gradient at the current iterate
- $B_k$ is a symmetric positive definite matrix approximating the Hessian
- $p$ is a step direction vector from $x_k$

### 3.2 Properties of the Quadratic Model
At the point $p = 0$ (corresponding to the current iterate $x_k$), the model satisfies:
- $m_k(0) = f_k$ (function value match)
- $\nabla m_k(0) = \nabla f_k$ (gradient match)

## 4. Finding the Search Direction

### 4.1 Minimizer of the Quadratic Model
To find the minimizer $p_k$ of our quadratic model (the step that minimizes $m_k(p)$), we differentiate $m_k(p)$ with respect to $p$ and set it to zero:

$$\nabla m_k(p) = \nabla f_k + B_k p$$

Setting $\nabla m_k(p_k) = 0$:

$$\nabla f_k + B_k p_k = 0$$

Solving for $p_k$:

$$B_k p_k = -\nabla f_k$$

Since $B_k$ is positive definite (and therefore invertible), we get:

$$p_k = -B_k^{-1} \nabla f_k \quad (6.2)$$

Where:
- $p_k$ is the step direction at iteration $k$
- $B_k^{-1}$ is the inverse of the Hessian approximation matrix

### 4.2 Descent Property
The direction $p_k$ is guaranteed to be a descent direction because:

$$\nabla f_k^T p_k = -\nabla f_k^T B_k^{-1} \nabla f_k < 0$$

The inequality holds because $B_k^{-1}$ is positive definite when $B_k$ is positive definite, making $\nabla f_k^T B_k^{-1} \nabla f_k > 0$ for any non-zero gradient.

## 5. Iteration and Update

### 5.1 Next Iterate
We use the search direction $p_k$ to compute the next iterate:

$$x_{k+1} = x_k + \alpha_k p_k \quad (6.3)$$

Where:
- $x_{k+1}$ is the next iterate (point) in the optimization process
- $\alpha_k$ is the step length (positive scalar) determined by a line search
- $p_k$ is the search direction computed in equation (6.2)

### 5.2 Updating the Quadratic Model
At the new iterate $x_{k+1}$, we construct a new quadratic model:

$$m_{k+1}(p) = f_{k+1} + \nabla f_{k+1}^T p + \frac{1}{2}p^T B_{k+1} p$$

Where:
- $m_{k+1}(p)$ is our new quadratic model at iteration $k+1$
- $f_{k+1} = f(x_{k+1})$ is the function value at the new iterate
- $\nabla f_{k+1} = \nabla f(x_{k+1})$ is the gradient at the new iterate
- $B_{k+1}$ is the updated Hessian approximation we need to determine
- $p$ is now a step direction vector from $x_{k+1}$

In this model, the variable $p$ represents a step from $x_{k+1}$:
- When $p = 0$, we're at $x_{k+1}$ (no step)
- When $p = -\alpha_k p_k$, we're at $x_k$ (stepping backward)

## 6. Determining $B_{k+1}$: The Secant Equation

### 6.1 Gradient Matching Conditions
We want our new model to match the gradients at both $x_{k+1}$ and $x_k$:

1. At $x_{k+1}$ (where $p = 0$):
   $$\nabla m_{k+1}(0) = \nabla f_{k+1}$$
   This is automatically satisfied by the definition of our model.

2. At $x_k$ (where $p = -\alpha_k p_k$):
   $$\nabla m_{k+1}(-\alpha_k p_k) = \nabla f_k$$

### 6.2 Derivation of the Secant Equation
From the definition of our model, we have:
$$\nabla m_{k+1}(p) = \nabla f_{k+1} + B_{k+1}p$$

Substituting $p = -\alpha_k p_k$ for the condition at $x_k$:
$$\nabla m_{k+1}(-\alpha_k p_k) = \nabla f_{k+1} + B_{k+1}(-\alpha_k p_k) = \nabla f_k$$

Rearranging:
$$\nabla f_{k+1} - \alpha_k B_{k+1} p_k = \nabla f_k$$

Further rearrangement gives us:
$$B_{k+1}\alpha_k p_k = \nabla f_{k+1} - \nabla f_k \quad (6.4)$$

This is known as the secant equation. It ensures that $B_{k+1}$ captures the curvature information between the two points.

## 7. Implications and Common Notation

### 7.1 Standard Notation
In optimization literature, the following shorthand notation is often used:
- $s_k = x_{k+1} - x_k = \alpha_k p_k$ (the step taken)
- $y_k = \nabla f_{k+1} - \nabla f_k$ (the gradient difference)

Using this notation, the secant equation (6.4) becomes:
$$B_{k+1}s_k = y_k$$

This means our Hessian approximation should capture how the gradient changes when we move from $x_k$ to $x_{k+1}$.

### 7.2 Additional Conditions
The secant equation alone is insufficient to uniquely determine $B_{k+1}$. Additional conditions are needed, which leads to the specific update formulas for different quasi-Newton methods such as BFGS and DFP.

In the case of BFGS, we impose additional conditions:
- Symmetry: $B_{k+1} = B_{k+1}^T$
- Minimum change from $B_k$ (in a weighted Frobenius norm)
- Positive definiteness preservation

These conditions together lead to the complete BFGS update formula:

$B_{k+1} = B_k - \frac{B_k s_k s_k^T B_k}{s_k^T B_k s_k} + \frac{y_k y_k^T}{y_k^T s_k}$

### 7.3 Comparison with Newton's Method
The key differences between Newton's method and BFGS are:

1. **Computational Efficiency**: 
   - Newton: Computes the exact Hessian at each iteration (expensive)
   - BFGS: Approximates the Hessian through low-rank updates (efficient)

2. **Memory Requirements**:
   - Both methods need $O(n^2)$ storage for an $n$-dimensional problem
   - Limited-memory variants (L-BFGS) reduce this to $O(n)$

3. **Convergence**:
   - Newton: Quadratic convergence near the solution
   - BFGS: Superlinear convergence (slower than Newton but still fast)

4. **Robustness**:
   - Newton: Can fail if the Hessian is not positive definite
   - BFGS: Maintains positive definiteness by construction

These properties make BFGS one of the most popular optimization methods, especially for problems where computing exact second derivatives is impractical.


## 8. BFGS Algorithm: Step-by-Step Implementation

The complete BFGS algorithm consists of the following steps that are performed in each iteration:

### 8.1 Algorithm Overview

**Initialization:**
- Start with initial point $x_0$
- Compute initial gradient $\nabla f_0 = \nabla f(x_0)$
- Initialize Hessian approximation $B_0$ (often set to identity matrix $I$)
- Set iteration counter $k = 0$

**Main Loop:**
1. **Compute search direction**: $p_k = -B_k^{-1} \nabla f_k$
2. **Perform line search**: Find step length $\alpha_k$ that satisfies Wolfe conditions
3. **Update position**: $x_{k+1} = x_k + \alpha_k p_k$
4. **Compute new gradient**: $\nabla f_{k+1} = \nabla f(x_{k+1})$
5. **Define vectors**:
   - $s_k = x_{k+1} - x_k = \alpha_k p_k$ (position difference)
   - $y_k = \nabla f_{k+1} - \nabla f_k$ (gradient difference)
6. **Update Hessian approximation**:
   $B_{k+1} = B_k - \frac{B_k s_k s_k^T B_k}{s_k^T B_k s_k} + \frac{y_k y_k^T}{y_k^T s_k}$
7. **Increment counter**: $k = k + 1$
8. **Check convergence**: If $\|\nabla f_k\| < \epsilon$, stop; otherwise return to step 1

### 8.2 Practical Implementation Details

1. **Computing $B_k^{-1}$ directly**: Rather than computing $B_k$ and then inverting it, many implementations directly update the inverse Hessian approximation $H_k = B_k^{-1}$. The update formula is:

   $H_{k+1} = \left(I - \frac{s_k y_k^T}{y_k^T s_k}\right) H_k \left(I - \frac{y_k s_k^T}{y_k^T s_k}\right) + \frac{s_k s_k^T}{y_k^T s_k}$

2. **Ensuring positive definiteness**: If $y_k^T s_k \leq 0$, which can happen with an insufficient line search, the BFGS update may not maintain positive definiteness. In practice, the update is skipped in this case.

3. **Line search**: The Wolfe conditions typically used are:
   - Sufficient decrease: $f(x_k + \alpha_k p_k) \leq f(x_k) + c_1 \alpha_k \nabla f_k^T p_k$
   - Curvature condition: $|\nabla f(x_k + \alpha_k p_k)^T p_k| \leq c_2 |\nabla f_k^T p_k|$
   
   where $0 < c_1 < c_2 < 1$ are constants (often $c_1 = 10^{-4}$ and $c_2 = 0.9$).

4. **Stopping criteria**: Common stopping criteria include:
   - Gradient norm: $\|\nabla f_k\| < \epsilon$
   - Function value change: $|f_k - f_{k-1}| < \epsilon$
   - Step size: $\|s_k\| < \epsilon$
   - Maximum iterations reached

### 8.3 Summary of Iterative Process

In each iteration, BFGS:
1. Uses the current Hessian approximation to determine a search direction
2. Performs a line search along this direction
3. Updates the position and computes the new gradient
4. Updates the Hessian approximation based on the position and gradient changes

The key to BFGS is the Hessian update formula, which incorporates new curvature information at each step while maintaining positive definiteness. This allows the algorithm to build an increasingly accurate approximation of the true Hessian as it iterates, leading to superlinear convergence near the solution.# Newton and BFGS Methods: A Systematic Derivation